<a href="https://colab.research.google.com/github/infinitywoke/AngelasBlogWithUsersandFlask/blob/main/Final_Version_Youtube_Automation_for_Free.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1 — Install Dependencies (Upgraded)
# ============================================================
# 1. Install Deno
!curl -fsSL https://deno.land/install.sh | sh

# 2. Install system dependencies
!apt-get install -qq -y ffmpeg

# 3. Install uv, then use it for blazing-fast package resolution
!pip install -q uv
!uv pip install -q --system \
    google-genai \
    "yt-dlp[default,curl-cffi]" \
    ffmpeg-python \
    pydantic \
    torch \
    torchaudio

print("✅ Dependencies installed.")

######################################################################## 100.0%
Archive:  /root/.deno/bin/deno.zip
  inflating: /root/.deno/bin/deno    
Deno was installed successfully to /root/.deno/bin/deno
sh: 105: cannot open /dev/tty: No such device or address
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 93.4 MB/s eta 0:00:00
✅ Dependencies installed.


In [24]:
# ============================================================
# CELL 2 — Configuration & API setup
# ============================================================

import os
from google.colab import userdata
from google import genai

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY_1")
except Exception:
    GEMINI_API_KEY = ""   # Paste your key here if Colab Secrets fails

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY not found. Add it via Colab Secrets.")

client = genai.Client(api_key=GEMINI_API_KEY)

# ── The Two-Tier Strategy ──
DRAFT_MODEL = "gemini-3.5-flash"
EDITOR_MODEL = "gemini-3.5-flash"

OUTPUT_DIR = "/content/bapamas_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── RAG Dictionary for DNT Guardrails ──
# Giving the model definitions stops context hallucination during translation.
DNT_DICT = {
    "tup": "pure clarified butter/ghee used for roasting",
    "Chiroti rava": "very fine semolina, distinct from thick rava",
    "Bapama": "grandmother (respectful term)",
    "solkadhi": "a cooling Konkani drink made from coconut milk and kokum",
    "kokum": "a souring agent used in coastal Indian cooking",
    "teekhein": "a spicy curry base",
    "ambat": "a tangy curry",
    "gashi": "a thick coconut-based curry",
    "saasam": "a sweet and spicy mustard-flavored dish",
    "uddamethi": "a black gram and fenugreek flavored curry",
    "phanna": "seasoning or tempering",
    "tadka": "tempering of spices in hot oil/ghee",
    "Panchakajjaya": "a traditional GSB sweet offering made of five ingredients",
    "Yelakki": "cardamom or small bananas",
    "Konkani": "the language and culture of the coastal region",
    "GSB": "Gowd Saraswat Brahmin community"
}

print("✅ Configuration and RAG Dictionary loaded.")

✅ Configuration and RAG Dictionary loaded.


In [3]:
# ============================================================
# CELL 3 — Universal Smart Downloader & Dimensional Preprocessing
# ============================================================

import subprocess
import os
import shutil
import ffmpeg
from pathlib import Path

# ============================================================
# UPDATED SNIPPET FOR CELL 3 — Hardcoded No-Cookie Downloader
# ============================================================

import subprocess
import os
from pathlib import Path

VIDEO_URL = "https://www.youtube.com/watch?v=Lc8YkgWQucY"
RAW_VIDEO_PATH = f"{OUTPUT_DIR}/source_video_raw.mp4"
OPTIMIZED_VIDEO_PATH = f"{OUTPUT_DIR}/optimized_video.mp4"

print(f"⬇ Downloading with advanced TLS spoofing: {VIDEO_URL}\n")

ydl_cmd = [
    "yt-dlp",
    "--format", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best",
    "--merge-output-format", "mp4",
    "--output", RAW_VIDEO_PATH,
    "--no-playlist",
    "--quiet",
    "--progress",

    # --- THE HARDCODED BYPASS ARMOR ---
    "--rm-cache-dir", # Clears old broken tokens
    "--impersonate", "chrome", # Uses curl-cffi to perfectly spoof a Chrome browser's TLS fingerprint
    "--extractor-args", "youtube:player_client=android_creator,android,ios", # Forces mobile API endpoints
    "--sleep-requests", "1", # Prevents aggressive rate-limiting

    VIDEO_URL
]

subprocess.run(ydl_cmd, capture_output=False)

if not Path(RAW_VIDEO_PATH).exists():
    raise FileNotFoundError("❌ Download failed. YouTube may have issued a hard IP block.")
else:
    print("✅ Raw video successfully downloaded without cookies.")

# ... [Continue to Phase 2: FFmpeg token-efficiency preprocessing] ...


⬇ Downloading with advanced TLS spoofing: https://www.youtube.com/watch?v=Lc8YkgWQucY

✅ Raw video successfully downloaded without cookies.


In [14]:
# ============================================================
# UPDATED CELL 4 — High-Precision Anchored Speaker Diarization
# ============================================================

import time
import sys
import ffmpeg
import json
from google.genai import types
from pydantic import BaseModel, Field

AUDIO_PATH = f"{OUTPUT_DIR}/audio_only.m4a"
REF_BAPAMA_PATH = f"{OUTPUT_DIR}/grandma.wav"
REF_ASHWIN_PATH = f"{OUTPUT_DIR}/me.wav"

# 1. Define the Strict Schema for the Ground Truth Transcript
class TranscriptLine(BaseModel):
    timestamp: str = Field(description="Strict format: H:MM:SS")
    speaker: str = Field(description="Bapama | Ashwin | Unknown")
    spoken_words: str = Field(description="The exact words spoken. Use [UNINTELLIGIBLE] if unclear.")

class GroundTruthTranscript(BaseModel):
    transcript: list[TranscriptLine]

# --- Audio Extraction Logic (Keep your existing FFmpeg code here) ---
print("🎵 Extracting audio track using T4 Hardware Acceleration...")
try:
    (
        ffmpeg
        .input(RAW_VIDEO_PATH, hwaccel='cuda')
        .audio
        .output(AUDIO_PATH, acodec='aac')
        .run(overwrite_output=True, capture_stdout=True, capture_stderr=True)
    )
    print("✅ Audio track isolated and extracted successfully.")
except ffmpeg.Error as e:
    print("\n❌ FFmpeg Audio Extraction Failed! Hidden error logs below:")
    print(e.stderr.decode())
    raise e
# -------------------------------------------------------------------

print("\n📝 Uploading Reference Audio and Main Track...")

# 2. Upload Files to Gemini
bapama_ref = client.files.upload(file=REF_BAPAMA_PATH, config={'display_name': 'bapama_voice_ref'})
ashwin_ref = client.files.upload(file=REF_ASHWIN_PATH, config={'display_name': 'ashwin_voice_ref'})
main_audio = client.files.upload(file=AUDIO_PATH, config={'display_name': 'bapamas_main_audio'})

print("⏳ Generating Zero-Hallucination Diarized Transcript...")

# 3. The Precision Prompt
diarization_prompt = """
You are an expert audio transcriber and linguist. I have provided three audio files in sequence:
1. Reference sample of Bapama's voice.
2. Reference sample of Ashwin's voice.
3. The main video audio track.

Listen to the first two files to learn their distinct vocal signatures. Then, transcribe the third file.

=== STRICT ANTI-HALLUCINATION RULES ===
1. NO TRANSLATION: You must transcribe exactly what is spoken. If they speak Konkani, transcribe it phonetically in the Latin alphabet. Do NOT translate it to English.
2. NO GUESSING: If a word is muffled by kitchen noise, write [UNINTELLIGIBLE]. Do not guess.
3. SPEAKER IDENTITY: If you cannot match the voice to the references with 100% certainty, label the speaker as "Unknown".
4. NO FILLER: Ignore stutters, "ums", and "ahs".
"""

# 4. Execute with Zero Creativity Parameters
transcript_response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=[
        bapama_ref,
        ashwin_ref,
        main_audio,
        diarization_prompt
    ],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=GroundTruthTranscript,
        temperature=0.0, # 🔥 Absolute factual enforcement
        top_k=1,         # 🔥 Eliminates alternative token guessing
    )
)

# 5. Extract and Format for the Next Pipeline Stage
if transcript_response.parsed and transcript_response.parsed.transcript:
    # Convert the strict JSON back into a readable block for the next cell's prompt
    parsed_lines = transcript_response.parsed.transcript
    RAW_TRANSCRIPT = "\n".join([f"[{line.timestamp}] {line.speaker}: {line.spoken_words}" for line in parsed_lines])
    print("✅ High-Precision Diarized Transcript Anchor generated.")
else:
    raise ValueError("❌ Failed to parse ground truth transcript into schema.")

print("\n⬆ Uploading token-optimized video to Gemini...")
video_file = client.files.upload(file=RAW_VIDEO_PATH, config={'display_name': 'bapamas_video_opt'})

print("⏳ Waiting for backend multimedia processing...")
start_time = time.time()
counter = 0

while video_file.state == "PROCESSING":
    counter += 1
    progress = min(98, int((1 - (0.85 ** counter)) * 100))
    elapsed = time.time() - start_time
    bar_length = 30
    filled_length = int(bar_length * progress // 100)
    bar = '█' * filled_length + '░' * (bar_length - filled_length)
    sys.stdout.write(f"\r🔍 [{bar}] {progress}% | Step: Ingestion Processing | Time: {elapsed:.1f}s")
    sys.stdout.flush()
    time.sleep(4)
    video_file = client.files.get(name=video_file.name)

if video_file.state == "ACTIVE" or str(video_file.state) == "FileState.ACTIVE":
    elapsed = time.time() - start_time
    bar = '█' * 30
    sys.stdout.write(f"\r✨ [{bar}] 100% | Status: File Ready!           | Total Time: {elapsed:.1f}s\n")
    sys.stdout.flush()
    print(f"🔗 Video URI: {video_file.uri}")
else:
    print(f"\n❌ Pipeline initialization halted. File status returned: {video_file.state}")

🎵 Extracting audio track using T4 Hardware Acceleration...
✅ Audio track isolated and extracted successfully.

📝 Uploading Reference Audio and Main Track...
⏳ Generating Zero-Hallucination Diarized Transcript...
✅ High-Precision Diarized Transcript Anchor generated.

⬆ Uploading token-optimized video to Gemini...
⏳ Waiting for backend multimedia processing...
✨ [██████████████████████████████] 100% | Status: File Ready!           | Total Time: 28.9s
🔗 Video URI: https://generativelanguage.googleapis.com/v1beta/files/6pt68ouri6d4


In [ ]:
!cp /content/bapamas_output/source_video_raw.mp4 /content/bapamas_output/optimized_video.mp4

In [15]:
# ============================================================
# CELL 5 — Pydantic Schemas & Anchored Multimodal Prompt
# ============================================================

import json
from pydantic import BaseModel, Field, model_validator
from typing import List, Optional

# 1. Deterministic Structural Validation with Visual Proof Requirement
class SubtitleLine(BaseModel):
    start: str = Field(description="Strict format: H:MM:SS.mmm")
    end: str = Field(description="Strict format: H:MM:SS.mmm")
    speaker: str = Field(description="Bapama | Ashwin | Both | None")
    action: Optional[str] = Field(description="Brief description of the visual action. Leave null if none.")
    action_timestamp: Optional[str] = Field(description="Strict format: H:MM:SS. The exact second the visual action happens. Null if action is null.")
    placement: str = Field(description="top | bottom")
    original: str = Field(description="Exact spoken words")
    translated: str = Field(description="English translation")

    @model_validator(mode='after')
    def check_time_logic(self):
        def _to_ms(ts):
            ts = ts.replace(',', '.')
            try:
                h, m, s_ms = ts.split(':')
                s, ms = s_ms.split('.')
                return int(h) * 3600000 + int(m) * 60000 + int(s) * 1000 + int(ms)
            except Exception:
                return 0

        start_ms = _to_ms(self.start)
        end_ms = _to_ms(self.end)

        if end_ms <= start_ms:
            raise ValueError(f"Temporal Logic Error: end time ({self.end}) must be greater than start time ({self.start}).")
        return self

class VideoExtraction(BaseModel):
    subtitles: List[SubtitleLine]

# 2. Anchored Prompting with Few-Shot Examples
def build_draft_prompt(start_time_str: str, end_time_str: str) -> str:
    dnt_context = "\n".join([f"- **{k}**: {v}" for k, v in DNT_DICT.items()])

    few_shot_examples = [
        {
            "start": "0:01:12.000",
            "end": "0:01:15.500",
            "speaker": "Bapama",
            "action": "Pouring ghee into the hot pan.",
            "action_timestamp": "0:01:13",
            "placement": "bottom",
            "original": "Ata he tup ghaluya.",
            "translated": "Now, let's add the tup (ghee)."
        },
        {
            "start": "0:01:16.000",
            "end": "0:01:18.000",
            "speaker": "Ashwin",
            "action": None,
            "action_timestamp": None,
            "placement": "top",
            "original": "Is the pan hot enough?",
            "translated": "Is the pan hot enough?"
        }
    ]
    examples_str = json.dumps(few_shot_examples, indent=2)

    return f"""
You are a MULTIMODAL Transcriber and Spatial Analyst working on a high-end culinary series.
You must WATCH the video carefully between {start_time_str} and {end_time_str}.
Use ABSOLUTE timestamps from the very beginning of the video.

=== 1. TEMPORAL ANCHOR (GROUND TRUTH) ===
Align your visual observations to these exact words. Do not invent dialogue:
{RAW_TRANSCRIPT}

=== 2. SPATIAL PLACEMENT RULES (CRITICAL) ===
You must analyze the visual framing to determine the 'placement' of the subtitle:
- Default to "bottom".
- If a crucial culinary action (e.g., hands mixing ingredients, close-up of a pan, adding spices) is occurring in the lower third of the frame, you MUST set placement to "top" so the text does not obscure the food.

=== 3. LINGUISTIC & CULTURAL INTEGRITY ===
Keep the following terms in their original form. Use these definitions to understand the context, and add a brief English gloss in parentheses ONLY on the first use.
{dnt_context}

=== STRICT OUTPUT EXAMPLES ===
If you document an 'action', you MUST provide the exact 'action_timestamp'.
Notice how the placement shifts to "top" in the example because the action is happening at the bottom of the frame.
{examples_str}
"""

print("✅ Cell 5 loaded: High-Precision Pydantic schemas and Prompts initialized.")

✅ Cell 5 loaded: High-Precision Pydantic schemas and Prompts initialized.


In [17]:
# ============================================================
# CELL 6 — Synchronous & High-Precision Drafting (No VAD)
# ============================================================

import math
import subprocess
from datetime import datetime
from google.genai import types
from tenacity import retry, wait_exponential, stop_after_attempt
from google.genai.errors import APIError
import time # Ensure time is imported at the top of Cell 6

def format_seconds(secs: float) -> str:
    m, s = divmod(int(secs), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

# 1. Synchronous FFprobe to get total video duration
def get_video_duration(filepath: str) -> float:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        filepath
    ]
    output = subprocess.check_output(cmd)
    return float(output.decode().strip())

# 2. Strict Zero-Hallucination Synchronous API Call
@retry(wait=wait_exponential(multiplier=2, min=5, max=15), stop=stop_after_attempt(5))
def safe_generate(chunk_prompt: str):
    try:
        return client.models.generate_content(
            model=DRAFT_MODEL,
            contents=[video_file, chunk_prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=VideoExtraction,
                temperature=0.0,
                top_k=1,
            )
        )
    except APIError as e:
        print(f"\n🚨 GOOGLE API ERROR UNMASKED: {e.code} - {e.message}")
        raise e

# 3. Synchronous Chunk Processor
def process_video_chunk(start_sec: float, end_sec: float) -> list:
    start_str = format_seconds(start_sec)
    end_str = format_seconds(end_sec)

    print(f"    ⏳ Processing [{start_str} to {end_str}]...")
    chunk_prompt = build_draft_prompt(start_str, end_str)

    try:
        response = safe_generate(chunk_prompt)
        subs = response.parsed.subtitles if response.parsed else []
        subs_dict = [sub.model_dump() for sub in subs]
        print(f"    ✅ Extracted {len(subs_dict)} lines for [{start_str} to {end_str}].")
        return subs_dict
    except Exception as e:
        print(f"    ❌ Chunk processing aborted: {e}")
        return []

# 4. Main Synchronous Execution Loop
def main_loop():
    print(f"🎬 Starting High-Precision Synchronous Loop for {DRAFT_MODEL}…")

    duration = get_video_duration(OPTIMIZED_VIDEO_PATH)
    CHUNK_SIZE_SEC = 180
    OVERLAP_SEC = 15
    total_chunks = math.ceil(duration / CHUNK_SIZE_SEC)

    print(f"    Video Duration : {format_seconds(duration)}")
    print("─" * 60)

    MASTER_SUBTITLES = []

    # Process sequentially, one by one
    for i in range(total_chunks):
        start_sec = max(0, (i * CHUNK_SIZE_SEC) - (OVERLAP_SEC if i > 0 else 0))
        end_sec = min(duration, start_sec + CHUNK_SIZE_SEC + OVERLAP_SEC)

        subs = process_video_chunk(start_sec, end_sec)
        MASTER_SUBTITLES.extend(subs)

        # 🛑 THE FIX: Force the script to cool down before hitting the API again
        if i < total_chunks - 1:
            print("    ⏳ Cooling down for 15 seconds to respect API Rate Limits...")
            time.sleep(15)

    # Deduplicate overlapping boundaries safely
    seen_starts = set()
    unique_subs = []
    for sub in MASTER_SUBTITLES:
        if sub['start'] not in seen_starts:
            unique_subs.append(sub)
            seen_starts.add(sub['start'])

    global RESULTS, RUN_TIMESTAMP
    RESULTS = {"subtitles": sorted(unique_subs, key=lambda x: x['start'])}
    RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("─" * 60)
    print(f"🎉 Drafting Complete! Total unique subtitles: {len(RESULTS['subtitles'])}")

# Trigger the loop
main_loop()

🎬 Starting High-Precision Synchronous Loop for gemini-3.5-flash…
    Video Duration : 00:21:38
────────────────────────────────────────────────────────────
    ⏳ Processing [00:00:00 to 00:03:15]...
    ✅ Extracted 67 lines for [00:00:00 to 00:03:15].
    ⏳ Cooling down for 15 seconds to respect API Rate Limits...
    ⏳ Processing [00:02:45 to 00:06:00]...
    ✅ Extracted 68 lines for [00:02:45 to 00:06:00].
    ⏳ Cooling down for 15 seconds to respect API Rate Limits...
    ⏳ Processing [00:05:45 to 00:09:00]...
    ✅ Extracted 63 lines for [00:05:45 to 00:09:00].
    ⏳ Cooling down for 15 seconds to respect API Rate Limits...
    ⏳ Processing [00:08:45 to 00:12:00]...
    ✅ Extracted 83 lines for [00:08:45 to 00:12:00].
    ⏳ Cooling down for 15 seconds to respect API Rate Limits...
    ⏳ Processing [00:11:45 to 00:15:00]...
    ✅ Extracted 48 lines for [00:11:45 to 00:15:00].
    ⏳ Cooling down for 15 seconds to respect API Rate Limits...
    ⏳ Processing [00:14:45 to 00:18:00]...
 

In [18]:
# ============================================================
# CELL 7 — Synchronous Sequential Editor
# ============================================================

import time
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from google.genai import types

class RefinedSubtitleItem(BaseModel):
    start: str
    end: str
    speaker: str
    translated: str = Field(description="Polished and formatted text. No speaker tags.")
    action: Optional[str] = Field(default=None)
    placement: Optional[str] = Field(default=None)

class RefinedSubtitlesContainer(BaseModel):
    refined_subtitles: List[RefinedSubtitleItem]

def build_multimodal_editor_prompt(batch_to_edit: list[dict], previous_context: list[dict]) -> str:
    import json
    batch_json = json.dumps(batch_to_edit, ensure_ascii=False, indent=2)
    context_json = json.dumps(previous_context, ensure_ascii=False, indent=2) if previous_context else "[]"
    batch_start = batch_to_edit[0].get('start', '0:00:00') if batch_to_edit else '0:00:00'
    batch_end = batch_to_edit[-1].get('end', '0:00:00') if batch_to_edit else '0:00:00'

    return f"""
You are a Senior Subtitle Editor and Timing Specialist adhering to Netflix Timed Text standards.
You must return a strict JSON payload. Watch from {batch_start} to {batch_end}.

=== 1. TEMPORAL PLACEMENT & PACING (CRITICAL) ===
You have the authority to subtly adjust the 'start' and 'end' timestamps of the input batch to maximize viewer immersion:
- PUNCHLINE PROTECTION: If a speaker reveals an ingredient, a joke, or a key instruction, ensure the 'start' time does NOT precede the audio cue. The text must appear exactly as the word is spoken, not before.
- READING EXTENSIONS: If a speaker talks very rapidly, slightly extend the 'end' time into the subsequent silence (by 500ms to 1 second) to allow the viewer enough time to finish reading.
- CONTINUOUS THOUGHTS: If a single sentence is split across two subtitle blocks, ensure the time gap between them is seamless so the viewer's reading rhythm is not broken.

=== 2. BROADCAST FORMATTING RULES (ABSOLUTE) ===
Your primary job is to format the 'translated' field correctly:
- LINE LIMIT: Maximum 2 lines per subtitle.
- CHARACTER LIMIT: Maximum 42 characters per line.
- SYNTACTIC BREAKS: If a subtitle requires two lines, you MUST break the text at a logical linguistic juncture.
    * DO NOT separate a noun from its adjective (e.g., "fine / semolina").
    * DO NOT separate a preposition from its phrase (e.g., "in the / pan").
    * Break at punctuation marks, before conjunctions (and, but, or), or before prepositional phrases.
- CONCISENESS: Strip all filler words ("um", "uh", "you know"). Subtitles must be easily readable at a quick glance.

=== 2. CHARACTER & TONE MAPPING ===
Maintain an immersive, generational kitchen dynamic:
- "Ashwin": Translate his dialogue to sound modern, inquisitive, and respectful. Use standard, contemporary English phrasing.
- "Bapama": Translate her dialogue to sound warm, authoritative, and steeped in traditional culinary wisdom. Her English should feel natural but carry the cadence of an experienced grandmother leading a kitchen.

=== 3. CONTEXTUAL CONTINUITY ===
Read the 'PREVIOUS CONTEXT' below. Ensure that questions match answers and that ongoing sentences flow seamlessly from the previous subtitle block.

=== 4. DATA SANITIZATION ===
Do NOT include speaker tags ([Bapama] or [Ashwin]) inside the 'translated' field.

=== PREVIOUS CONTEXT ===
{context_json}

=== SUBTITLES TO EDIT & TIME-SYNC ===
{batch_json}
"""

def process_editor_batch(batch_to_edit: list[dict], context: list[dict], depth: int = 0) -> list:
    if not batch_to_edit: return []
    prompt = build_multimodal_editor_prompt(batch_to_edit, context)

    try:
        # Switched to the synchronous client model
        resp = client.models.generate_content(
            model=EDITOR_MODEL,
            contents=[video_file, prompt],
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json",
                response_schema=RefinedSubtitlesContainer,
            )
        )

        if resp.parsed and resp.parsed.refined_subtitles:
            return [sub.model_dump() for sub in resp.parsed.refined_subtitles]
        raise ValueError("Failed to parse schema.")

    except Exception as e:
        # Fallback: Divide and conquer if the batch fails
        if len(batch_to_edit) > 1:
            mid = len(batch_to_edit) // 2
            time.sleep(2)

            h1 = process_editor_batch(batch_to_edit[:mid], context, depth + 1)
            new_ctx = h1[-5:] if h1 else context
            h2 = process_editor_batch(batch_to_edit[mid:], new_ctx, depth + 1)

            return h1 + h2

        return batch_to_edit

def main_editor_loop():
    global RESULTS
    raw_subs = RESULTS.get("subtitles", [])
    if not raw_subs:
        print("⚠ No subtitles found to edit.")
        return

    BATCH_SIZE = 45

    # Break raw_subs into sequential batches
    all_batches = [raw_subs[i:i + BATCH_SIZE] for i in range(0, len(raw_subs), BATCH_SIZE)]

    print(f"🎬 Starting Synchronous Editor... Processing {len(all_batches)} sequential batches.")
    print("─" * 60)

    REFINED_MASTER_LIST = []
    previous_context = []

    # Process batches one by one, rolling the context forward natively
    for i, batch in enumerate(all_batches):
        print(f"  📦 Processing Batch {i+1}/{len(all_batches)}...")
        batch_results = process_editor_batch(batch, previous_context)
        REFINED_MASTER_LIST.extend(batch_results)

        # Grab the last 5 polished lines to feed into the next batch
        previous_context = REFINED_MASTER_LIST[-5:] if REFINED_MASTER_LIST else []
        time.sleep(2) # Brief pause to respect API limits

    # Sort final output temporally just to be absolutely safe
    RESULTS["subtitles"] = sorted(REFINED_MASTER_LIST, key=lambda x: x.get('start', '0'))
    print("─" * 60)
    print(f"🎉 Editor Complete! {len(RESULTS['subtitles'])} polished subtitles ready.")

# Execute the loop
main_editor_loop()

🎬 Starting Synchronous Editor... Processing 12 sequential batches.
────────────────────────────────────────────────────────────
  📦 Processing Batch 1/12...
  📦 Processing Batch 2/12...
  📦 Processing Batch 3/12...
  📦 Processing Batch 4/12...
  📦 Processing Batch 5/12...
  📦 Processing Batch 6/12...
  📦 Processing Batch 7/12...
  📦 Processing Batch 8/12...
  📦 Processing Batch 9/12...
  📦 Processing Batch 10/12...
  📦 Processing Batch 11/12...
  📦 Processing Batch 12/12...
────────────────────────────────────────────────────────────
🎉 Editor Complete! 482 polished subtitles ready.


In [19]:
# ============================================================
# CELL 7 — SRT Sanitizer & Exporter
# ============================================================

import json
from pathlib import Path

BASE = Path(OUTPUT_DIR) / RUN_TIMESTAMP
BASE.mkdir(parents=True, exist_ok=True)
srt_file_path = BASE / "bapamas_subtitles_final.srt"

def time_to_ms(ts_str: str) -> int:
    ts_str = ts_str.replace(',', '.')
    h, m, s_ms = ts_str.split(':')
    s, ms = s_ms.split('.')
    return int(h) * 3600000 + int(m) * 60000 + int(s) * 1000 + int(ms)

def ms_to_srt(ms: int) -> str:
    h = ms // 3600000
    ms = ms % 3600000
    m = ms // 60000
    ms = ms % 60000
    s = ms // 1000
    ms = ms % 1000
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

try:
    raw_subtitles = RESULTS.get('subtitles', [])
    processed_subs = []

    for sub in raw_subtitles:
        text = (sub.get('translated') or sub.get('refined_subtitles') or
                sub.get('refined') or sub.get('text') or sub.get('original') or "").strip()

        if not text: continue

        start_ms = time_to_ms(sub.get('start', '00:00:00.000'))
        end_ms = time_to_ms(sub.get('end', '00:00:00.000'))

        # 🚨 FIX: The Time-Traveler Bug
        if end_ms <= start_ms:
            end_ms = start_ms + 2000

        processed_subs.append({
            "start_ms": start_ms,
            "end_ms": end_ms,
            "text": text
        })

    # 🚨 FIX: The Out-of-Order Bug
    processed_subs.sort(key=lambda x: x['start_ms'])

    # Format the final SRT strings
    srt_lines = []
    for i, sub in enumerate(processed_subs, start=1):
        start_str = ms_to_srt(sub['start_ms'])
        end_str = ms_to_srt(sub['end_ms'])

        srt_lines.append(f"{i}\n{start_str} --> {end_str}\n{sub['text']}\n")

    with open(srt_file_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(srt_lines))

    with open(BASE / "full_output.json", 'w', encoding='utf-8') as f:
        f.write(json.dumps(RESULTS, indent=2))

    print(f"✅ Extracted, Cleaned, and Sorted {len(processed_subs)} subtitles!")
    print(f"📁 Saved as: {srt_file_path}")

except Exception as e:
    print(f"❌ An error occurred during export: {e}")

✅ Extracted, Cleaned, Sorted, and Spatially Aligned 482 subtitles!
📁 Saved as: /content/bapamas_output/20260611_002108/bapamas_subtitles_final.srt


In [20]:
# ============================================================
# CELL 8 — Netflix Standard Temporal Exporter
# ============================================================

import json
import textwrap
from pathlib import Path

BASE = Path(OUTPUT_DIR) / RUN_TIMESTAMP
BASE.mkdir(parents=True, exist_ok=True)
srt_file_path = BASE / "bapamas_subtitles_netflix_timed.srt"

def time_to_ms(ts_str: str) -> int:
    ts_str = ts_str.replace(',', '.')
    h, m, s_ms = ts_str.split(':')
    s, ms = s_ms.split('.')
    return int(h) * 3600000 + int(m) * 60000 + int(s) * 1000 + int(ms)

def ms_to_srt(ms: int) -> str:
    ms = max(0, ms)
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{int(h):02d}:{int(m):02d}:{int(s):02d},{int(ms):03d}"

try:
    raw_subtitles = RESULTS.get('subtitles', [])
    processed_subs = []

    # 1. Parse and validate raw timings
    for sub in raw_subtitles:
        text = (sub.get('translated') or sub.get('refined_subtitles') or sub.get('text') or "").strip()
        if not text: continue

        start_ms = time_to_ms(sub.get('start', '00:00:00.000'))
        end_ms = time_to_ms(sub.get('end', '00:00:00.000'))

        processed_subs.append({
            "start_ms": start_ms,
            "end_ms": end_ms,
            "text": text
        })

    # Sort chronologically to prevent temporal anomalies
    processed_subs.sort(key=lambda x: x['start_ms'])

    # 2. Apply Netflix Mathematical Temporal Rules
    for i in range(len(processed_subs)):
        current = processed_subs[i]

        # Rule A: Minimum Duration (833ms / ~20 frames)
        if (current['end_ms'] - current['start_ms']) < 833:
            current['end_ms'] = current['start_ms'] + 833

        # Rule B: Maximum Duration (7 seconds)
        if (current['end_ms'] - current['start_ms']) > 7000:
            current['end_ms'] = current['start_ms'] + 7000

        # Look ahead to the next subtitle for gap/overlap management
        if i < len(processed_subs) - 1:
            next_sub = processed_subs[i+1]
            gap = next_sub['start_ms'] - current['end_ms']

            # Rule C: Overlap Resolution (End current 1ms before next starts)
            if gap < 0:
                current['end_ms'] = next_sub['start_ms'] - 1

            # Rule D: Flicker Prevention (If gap is < 84ms, snap them together)
            elif 0 <= gap <= 84:
                current['end_ms'] = next_sub['start_ms'] - 1

    # 3. Format and Wrap Text
    srt_lines = []
    for index, sub in enumerate(processed_subs, start=1):
        start_str = ms_to_srt(sub['start_ms'])
        end_str = ms_to_srt(sub['end_ms'])

        # Enforce the 42-character line limit natively
        wrapped_text = textwrap.fill(sub['text'], width=42)

        srt_lines.append(f"{index}\n{start_str} --> {end_str}\n{wrapped_text}\n")

    # 4. Write output
    with open(srt_file_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(srt_lines))

    print(f"✅ Netflix Temporal Rules Applied! Formatted {len(processed_subs)} subtitles.")
    print(f"📁 Saved to: {srt_file_path}")

except Exception as e:
    print(f"❌ An error occurred during export: {e}")

✅ Netflix Temporal Rules Applied! Formatted 482 subtitles.
📁 Saved to: /content/bapamas_output/20260611_002108/bapamas_subtitles_netflix_timed.srt
🚀 Synthesizing recipe, SEO metadata, and design prompts...
✅ YouTube Package successfully generated!
📁 Saved to: /content/bapamas_output/20260611_002108/youtube_optimization_package.md


In [25]:
# ============================================================
# CELL 9 — The Auto-Tagging YouTube Optimization Engine
# ============================================================

import time
import json
import re
from pathlib import Path
from google.genai import types

# Hardcoded Brand Name for the Delimiter Schema
BRAND = "Bapamas-Kitchen"

def generate_youtube_package() -> dict:
    # 1. Ensure the JSON payload exists on disk from Cell 8
    json_path = BASE / "full_output.json"
    if not json_path.exists():
        return {"error": "❌ Error: full_output.json not found. Please ensure Cell 8 completed successfully."}

    print("📄 Uploading timeline JSON to Gemini File API...")

    # 2. Upload the JSON file directly to Gemini
    timeline_file = client.files.upload(file=str(json_path), config={'display_name': 'video_timeline'})

    while timeline_file.state == "PROCESSING":
        time.sleep(2)
        timeline_file = client.files.get(name=timeline_file.name)

    # 3. The Upgraded & Highly Specific Synthesis Prompt
    synthesis_prompt = """
**Role:**
Act as an elite 2026 YouTube Culinary Algorithm Expert and SEO Specialist. Your goal is to optimize heritage/regional culinary content for a dual market (local Indian audience + global food enthusiasts) using the latest algorithmic signals, sociolinguistic strategies (code-switching/glocalization), and high-end F&B photography principles.

**Channel Context:**
Ensure the branding naturally reflects the hosts, Ashwin, the grandson, who records the recipes and the Grandma, Sharada, affectionately known as Bapama and the channel name "Bapama's Kitchen", a YouTube channel dedicated to sharing traditional and authentic GSB Konkani Culinary and Cultural Heritage with the world. Integrate this identity smoothly into the descriptions and tags to build consistent channel authority.

**Task:**
Analyze the attached JSON transcript file containing timestamps, speaker names, translated dialogue, and actions. Based *only* on this context, generate a comprehensive YouTube Optimization Package formatted cleanly in Markdown.

**Formatting & Output Requirements:**
You must output EXACTLY 9 sections in the following order.

**0. Semantic Metadata (CRITICAL INITIALIZATION)**
At the very beginning of your response, output a strict JSON object wrapped in <metadata> tags. This will be extracted by a script to name the database file. Do not include markdown codeblocks inside the tags.
Example:
<metadata>
{
  "type": "Recipe",
  "category": "Breakfast-Sweet",
  "specific_name": "Rava-Sheera"
}
</metadata>
- "type" MUST be one of: Recipe, Pairing, Strategy, Concept.
- "category": Broad category descriptor. Replace all spaces with hyphens.
- "specific_name": The specific dish or topic. Replace all spaces with hyphens.

**1. Detailed Description**
* **The SEO Hook:** The first 150 characters must contain the primary keyword, articulate a clear value proposition, and define the outcome.
* **Expanded Semantic Summary:** A ~200-word paragraph integrating "glocalized" language (blending native terminology with English descriptors). Establish cultural authority and integrate secondary long-tail keywords naturally.

**2. Ingredients**
* Extract or logically infer the ingredients mentioned in the transcript.
* Present them clearly, using both standard and metric measurements where applicable to appeal to a global audience.

**3. Instructions (with timestamps)**
* Extract the core actions from the transcript.
* Format them as steps and a chronological list using Step 1 (e.g. Step 1).

**4. Expert Tips (e.g., Bapama's Tips)**
* Highlight the specific generational techniques, temperature controls, or culinary secrets mentioned in the transcript.
* Explain *why* these steps are important to establish topical authority.

**5. Video Chapters**
* Create a chapter list.
* **Crucial Rule:** The first chapter MUST be "0:00 - Introduction to [Topic]".
* Write titles as SEO-friendly micro-headlines (e.g., "3:56 - Adding Fine Semolina (Rava) to Hot Ghee").

**6. Hashtags**
* Provide 5 to 7 hashtags combining niche cultural tags with broad global categorizations. Placed on a single line.

**7. YouTube Search Tags (Strict 500-Character Limit)**
* **Character Budget:** Output a maximum of 15 tags. The total character count (including spaces and commas) MUST remain under 450 characters to leave room for the user to append their channel name.
* **Bilingual & Phonetic Mapping (Crucial):** Identify regional ingredients/dishes in the transcript and force the inclusion of phonetic spellings and English equivalents.
* **Hierarchical Structure:**
    * **Tags 1-3 (Exact Match):** Direct reflection of the title and primary search intent.
    * **Tags 4-8 (Category Clusters):** Broader regional and niche identifiers.
    * **Tags 9-15 (Conversational & Misspellings):** High-intent phrases and common typos.
* **Negative Constraints (BANNED TAGS):** * Do NOT use single-word, generic tags ("food", "cooking", "India", "tasty", "recipe").
    * Every tag must be a phrase of at least two words.
    * Do NOT include competitor channel names.

**8. Thumbnail Generation Prompt (Midjourney v6)**
* Create a hyper-realistic image generation prompt using the F.O.C.A.L. framework (Format, Object, Camera, Atmosphere, Lighting).
* Choose ONE of the following aesthetics based on the food type:
    * *Dark, Moody "Fine Dining"* (for curries/fried food)
    * *Bright, Airy "Morning Kitchen"* (for breakfasts/lifestyle)
    * *Levitating Action* (for dynamic pouring/stirring/splashing).
* Include specifics like: props (hammered copper, banana leaf), lighting (strobe, rim light), camera specs (macro lens, shallow depth of field), and end with `--ar 16:9 --style raw --v 6.0`.

=== EXTRACTED VIDEO DATA ===
The raw transcript data is provided in the attached JSON file.
"""

    print("🚀 Synthesizing recipe, semantic metadata, and SEO package...")
    try:
        response = client.models.generate_content(
            model=EDITOR_MODEL,
            contents=[timeline_file, synthesis_prompt],
            config=types.GenerateContentConfig(temperature=0.2)
        )

        # Clean up the file from the API immediately
        client.files.delete(name=timeline_file.name)
        raw_text = response.text

        # --- 4. The Auto-Tagging Extraction Engine ---
        metadata = {
            "type": "Content",
            "category": "Uncategorized",
            "specific_name": "Video"
        }

        # Regex to find the <metadata> block
        match = re.search(r'<metadata>(.*?)</metadata>', raw_text, re.DOTALL)
        if match:
            try:
                # Clean up any potential markdown ticks
                clean_json_str = match.group(1).replace('```json', '').replace('```', '').strip()
                metadata = json.loads(clean_json_str)

                # Silently strip the metadata block from the final markdown text
                raw_text = raw_text.replace(match.group(0), "").strip()
            except json.JSONDecodeError:
                print("⚠ Regex caught the metadata block, but JSON parsing failed. Using fallbacks.")
        else:
            print("⚠ <metadata> tags not found in response. Using fallbacks.")

        return {
            "markdown": raw_text,
            "metadata": metadata
        }

    except Exception as e:
        return {"error": f"❌ Failed to generate package: {e}"}

# --- Execution & Autonomous Semantic Saving ---
if "RESULTS" in locals() and "subtitles" in RESULTS:
    output_data = generate_youtube_package()

    if "error" in output_data:
        print(output_data["error"])
    else:
        meta = output_data["metadata"]
        markdown_content = output_data["markdown"]

        # Construct the Semantic Filename automatically
        semantic_filename = f"{meta.get('type', 'Recipe')}_{meta.get('category', 'Misc')}_{meta.get('specific_name', 'Item')}_{BRAND}.md"
        package_file_path = BASE / semantic_filename

        # Save to disk
        with open(package_file_path, 'w', encoding='utf-8') as f:
            f.write(markdown_content)

        print(f"✅ Auto-Tagging Complete! YouTube Package successfully generated.")
        print(f"📁 Saved to: {package_file_path}\n")

        print("🧠 Vector Database Ready! Extracted Payload:")
        print(json.dumps({**meta, "brand": BRAND}, indent=2))
        print("─" * 60)
        print(markdown_content)
else:
    print("⚠ RESULTS dictionary not found. Please ensure drafting and exporting cells ran successfully.")

📄 Uploading timeline JSON to Gemini File API...
🚀 Synthesizing recipe, semantic metadata, and SEO package...
✅ Auto-Tagging Complete! YouTube Package successfully generated.
📁 Saved to: /content/bapamas_output/20260611_002108/Recipe_Breakfast-Sweet_Rava-Sheera_Bapamas-Kitchen.md

🧠 Vector Database Ready! Extracted Payload:
{
  "type": "Recipe",
  "category": "Breakfast-Sweet",
  "specific_name": "Rava-Sheera",
  "brand": "Bapamas-Kitchen"
}
────────────────────────────────────────────────────────────
### 1. Detailed Description

* **The SEO Hook:** Learn how to make the perfect, non-sticky Rava Sheera with Bapama's traditional GSB Konkani recipe for a rich, melt-in-your-mouth sweet breakfast. (149 characters)

* **Expanded Semantic Summary:** 
Welcome back to Bapama's Kitchen! In today's episode, Ashwin and his grandmother, Sharada (affectionately known as Bapama), share their treasured family recipe for traditional Rava Sheera (also known as Sooji Halwa or Kesari Bath). This classic G